# V59 - Pairwise MLP ranker

Pre-registered plan: pointwise regressors plateau, so test a pairwise RankNet-style MLP on feature differences with a Copeland score. SVD compresses the sparse view first. Sampling keeps informative pairs with a relevance gap. Same grouped folds under a fresh seed, same floor and worst-fold locks against the word-only baseline. SVD is unsupervised and fit on train only.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

SEED = 909
N_SPLITS = 5
GATE_FLOOR = 0.002
PAIRS_PER_USER = 24
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v59")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Compressed features and pair sampling

SVD to 256 dims plus decoded assessment. Pairs are sampled within user where relevance differs, both orientations for symmetry.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
T = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs)
svd = TruncatedSVD(n_components=256, random_state=SEED).fit(T[:n_train])
Z = np.hstack([svd.transform(T).astype(np.float32), A])
Z_tr, Z_te = Z[:n_train], Z[n_train:]
print("compressed:", Z.shape)

def sample_pairs(uidx, r, n=PAIRS_PER_USER):
    r = np.random.RandomState(SEED + uidx)
    ii, jj = np.triu_indices(len(MODS), 1)
    gap = np.abs(Y[uidx, ii] - Y[uidx, jj])
    elig = np.where(gap > 0.05)[0]
    if len(elig) == 0:
        return []
    out = []
    for e in r.choice(elig, size=min(n, len(elig)), replace=False):
        i, j = int(ii[e]), int(jj[e])
        out += [(i, j, 1.0 if Y[uidx, i] > Y[uidx, j] else 0.0),
                (j, i, 1.0 if Y[uidx, j] > Y[uidx, i] else 0.0)]
    return out

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

## Fold-wise pairwise training and Copeland scoring

One MLP per fold on standardized differences. Test scores are mean pairwise win probabilities. Baseline is word-only Ridge from the same folds.

In [ ]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def copeland(clf, scaler, Zq):
    S = np.zeros((len(Zq), len(MODS)))
    for i in range(len(MODS)):
        acc = np.zeros(len(Zq))
        for j in range(len(MODS)):
            if i == j:
                continue
            acc += clf.predict_proba(scaler.transform(Zq[:, i, :] - Zq[:, j, :]))[:, 1]
        S[:, i] = acc / (len(MODS) - 1)
    return S

Zmod_tr = np.stack([Z_tr] * len(MODS), axis=1)
Zmod_te = np.stack([Z_te] * len(MODS), axis=1)
oof_pair = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(Z_tr, strata, fold_groups)):
    D, L = [], []
    for u in tr:
        for i, j, lab in sample_pairs(int(u), None):
            D.append(Z_tr[u, :] * 0 + (Zmod_tr[u, i] - Zmod_tr[u, j]))
            L.append(lab)
    D, L = np.array(D), np.array(L)
    scaler = StandardScaler().fit(D)
    clf = MLPClassifier((256, 128), max_iter=30, random_state=SEED)
    clf.fit(scaler.transform(D), L)
    oof_pair[va] = copeland(clf, scaler, Zmod_tr[va])
    fids[va] = fi + 1
    print(f"fold {fi + 1}: pairs={len(L)} pair-train-acc={clf.score(scaler.transform(D), L):.4f} oof={ndcg5(Y[va], oof_pair[va]).mean():.5f}", flush=True)

oof_w = np.zeros_like(Y)
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()[:n_train]
for fi in range(1, N_SPLITS + 1):
    tr, va = np.where(fids != fi)[0], np.where(fids == fi)[0]
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0, random_state=SEED)
        m.fit(W[tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[va])
base = ndcg5(Y, oof_w)
d = ndcg5(Y, oof_pair) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_pair[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"pairwise OOF={ndcg5(Y, oof_pair).mean():.5f} baseline={base.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT pairwise" if (lo > GATE_FLOOR and wf >= 0) else "KEEP baseline; pairwise does not lift")